# CSE 5509 Final Project: Single-Camera BEV Scene Visualization

## 1. Setup
This setup works either in Colab or from a local checkout. The notebook runs dataset discovery, pretrained segmentation/depth/detection models, approximate BEV projection, and per-location compositing.

> This is an approximate visualization pipeline for course analysis, not a calibrated mapping system.



In [ ]:
from pathlib import Path
import sys
import json

# Only mount Drive when running in Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print(f"Drive mount skipped or unavailable: {e}")

# ---------------------------------------------------------------------
# IMPORTANT: set this to the folder that contains:
#   final-project-cse5509-v2.ipynb
#   bev_pipeline.py
#   data/
# ---------------------------------------------------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/CSE 5509 Final Project")

# If your folder is somewhere else, uncomment this search helper once:
# matches = list(Path("/content/drive/MyDrive").rglob("bev_pipeline.py"))
# print(matches[:20])

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"PROJECT_DIR does not exist: {PROJECT_DIR}")

if not (PROJECT_DIR / "bev_pipeline.py").exists():
    raise FileNotFoundError(
        f"Could not find bev_pipeline.py in {PROJECT_DIR}. "
        "Set PROJECT_DIR to the folder containing bev_pipeline.py."
    )

# Make local project module importable
sys.path.insert(0, str(PROJECT_DIR))

# Also change working directory so relative paths behave as expected
%cd "{PROJECT_DIR}"

from bev_pipeline import PipelineConfig, resolve_project_paths

paths = resolve_project_paths()

# Handle both possible layouts:
#   repo/data/loc1/...
#   repo/data/data/loc1/...
data_dir = paths["data_dir"]
if (data_dir / "data").exists():
    data_dir = data_dir / "data"

CONFIG = PipelineConfig(
    repo_root=PROJECT_DIR,
    data_dir=data_dir,
    output_dir=PROJECT_DIR / "outputs",
    run_small_demo=True,  # set False to process all images
    demo_locations=1,
    demo_images_per_location=2,
    assumed_hfov_deg=90.0,
    depth_clip_min=0.05,
    depth_clip_max=1.0,
    bev_max_distance_m=40.0,
    bev_scale_px_per_m=12.0,
    bev_height_px=720,
    bev_width_px=720,
    detection_threshold=0.5,
)

print("Repo root :", CONFIG.repo_root)
print("Data dir  :", CONFIG.data_dir)
print("Output dir:", CONFIG.output_dir)
print("bev_pipeline.py found:", (PROJECT_DIR / "bev_pipeline.py").exists())

## 2. Dataset discovery

Count the images from the data folders.


In [ ]:
from bev_pipeline import discover_dataset

summary = discover_dataset(CONFIG.data_dir)
print(json.dumps({
    'location_count': summary['location_count'],
    'total_images': summary['total_images'],
    'images_per_location': summary['images_per_location'],
}, indent=2))


## 3. Load pretrained models

The first run may download pretrained model weights.


In [ ]:
from bev_pipeline import default_model_states

DEVICE = 'cpu'
model_states = default_model_states(device=DEVICE)
print({k: v.get('available', False) for k, v in model_states.items()})
for k, v in model_states.items():
    if not v.get('available', False):
        print(f"[{k}] fallback reason: {v.get('reason')}")


## 4. Run the pipeline

The default configuration runs a small subset so the notebook can be checked quickly.


In [ ]:
from bev_pipeline import ensure_output_dirs, process_location

output_dirs = ensure_output_dirs(CONFIG.output_dir)

location_names = summary['locations']
if CONFIG.run_small_demo:
    location_names = location_names[:CONFIG.demo_locations]

run_report = []
for loc in location_names:
    image_paths = summary['files'][loc]
    if CONFIG.run_small_demo:
        image_paths = image_paths[:CONFIG.demo_images_per_location]
    print(f'Processing {loc}: {len(image_paths)} image(s)')
    res = process_location(loc, image_paths, CONFIG, model_states, output_dirs)
    run_report.append(res)

report_path = CONFIG.output_dir / 'run_report.json'
report_path.write_text(json.dumps(run_report, indent=2), encoding='utf-8')
print(f'Saved run report: {report_path}')


## 5. Sanity checks


In [ ]:
# Sanity checks requested by QA
assert summary['location_count'] > 0, 'No locations discovered'
assert summary['total_images'] > 0, 'No images discovered'

for loc_result in run_report:
    assert Path(loc_result['stitched_bev']).exists(), 'Missing stitched BEV output'
    for img_result in loc_result['images']:
        assert Path(img_result['bev_path']).exists(), 'Missing per-image BEV output'
        assert Path(img_result['det_path']).exists(), 'Missing detection overlay output'
        labels = [r['instance_label'] for r in img_result['instances']]
        assert len(labels) == len(set(labels)), 'Instance labels are not unique'

print('Sanity checks passed.')


## 6. Output files


In [ ]:
from pathlib import Path

for p in sorted((CONFIG.output_dir).glob('**/*')):
    if p.is_file():
        print(p.relative_to(CONFIG.repo_root))


## 7. Assumptions and limitations

- Monocular depth is relative, so distance values are approximate.
- Camera intrinsics are assumed from image size and horizontal FOV.
- Ground-plane projection is a simplification and may fail on slopes/elevation changes.
- Segmentation and detection can miss or misclassify difficult objects.
- Per-location compositing uses fixed-angle assumptions from directional image names.

For the final report, useful figures are the detection overlays, segmentation masks, per-image BEV images, and the per-location composite.
